# Lab 1: Automated Ingestion (Building Structured Knowledge)
In this lab, we will build a pipeline that reads a messy, unstructured PDF and uses Groq to automatically extract distinct concepts into clean Open Knowledge Format (OKF) files.

### Step - 1 Install required libraries

In [39]:
# !pip install PyPDF2 python-dotenv groq requests

### Step 2: Import Libraries

In [40]:
import os
import json
import PyPDF2
from dotenv import load_dotenv
from groq import Groq

### Step 3: Setup API Keys

In [41]:
load_dotenv("../.env")

# Initialize the Native Groq Client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

### Step 4: Locate and Read the Document

In [42]:
# Point to the local PDF (already in the root folder)
PDF_PATH = "../SunFactSheet.pdf"

if os.path.exists(PDF_PATH):
    print(f"Success: Found the document at '{PDF_PATH}'")
else:
    print(f"Error: Could not find the document.")

# Extract text
reader = PyPDF2.PdfReader(PDF_PATH)
raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text() + "\n"
    
print("Text extraction complete! Ready for AI processing.")

Success: Found the document at '../SunFactSheet.pdf'
Text extraction complete! Ready for AI processing.


### Step 5: Extract Concepts with Groq

In [45]:
system_prompt = """
You are a data extraction assistant. Read the text below and extract every distinct technical fact or value present in the source.
Do not limit yourself to a fixed number of facts — extract all of them, no matter how many there are.
Use the same wording as the source text for each fact — do not paraphrase or rename values.
Each fact must have its own separate concept entry, even if two facts are about a related topic.
Keep each "content" field short — 1 to 2 sentences maximum, stating only the fact and its value.
Respond in JSON only, matching this format:
{
  "concepts": [
    {
      "filename": "concept_name.md",
      "type": "concept",
      "title": "Human Readable Title",
      "tags": ["tag1", "tag2"],
      "description": "A single sentence summary",
      "content": "The full detailed explanation in markdown format."
    }
  ]
}
Output ONLY the JSON. No extra text, no markdown formatting blocks.
"""

print("Sending document to Groq for extraction...")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile", 
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": raw_text}
    ],
    temperature=0.0,
    max_tokens=8000
)

raw_json = response.choices[0].message.content.strip()

try:
    structured_data = json.loads(raw_json)
    concepts = structured_data.get("concepts", [])
    print(f"Success! AI identified {len(concepts)} distinct concepts.")
except json.JSONDecodeError:
    print("Error: The model's JSON response was cut off or malformed.")
    print("Try reducing the amount of source text, or check the raw output below:")
    print(raw_json[-500:])
    concepts = []

Sending document to Groq for extraction...
Success! AI identified 66 distinct concepts.


### Step 6: Build OKF Files and Update Index

In [46]:
# Prepare the shared output directory outside the lab folder
output_dir = "../output_wiki"
os.makedirs(output_dir, exist_ok=True)

index_path = os.path.join(output_dir, "index.md")

# Create master index if it doesn't exist
if not os.path.exists(index_path):
    with open(index_path, "w", encoding="utf-8") as f:
        f.write("# Master Index\n\n")

print(f"Output directory ready at: {output_dir}")

Output directory ready at: ../output_wiki


In [47]:
# Loop through the extracted data and create the markdown files
for concept in concepts:
    filename = concept['filename'].replace(" ", "_").lower()
    if not filename.endswith('.md'):
        filename += '.md'
        
    file_path = os.path.join(output_dir, filename)
    
    # Format the OKF content (Metadata Layer + Content Layer)
    okf_content = f"""type: {concept['type']}
title: {concept['title']}
tags: {concept['tags']}
description: {concept['description']}
---
# {concept['title']}

{concept['content']}
"""
    
    # Write the individual concept file
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(okf_content)
    
    print(f"Created: {filename}")
    
    # Safely append to the Master Index
    with open(index_path, "a", encoding="utf-8") as f:
        f.write(f"- **{filename}**: {concept['description']}\n")

print("\nPipeline complete! Ready for Lab 2.")

Created: sun_mass.md
Created: earth_mass.md
Created: sun_to_earth_mass_ratio.md
Created: sun_gm.md
Created: earth_gm.md
Created: sun_to_earth_gm_ratio.md
Created: sun_volume.md
Created: earth_volume.md
Created: sun_to_earth_volume_ratio.md
Created: sun_volumetric_mean_radius.md
Created: earth_volumetric_mean_radius.md
Created: sun_to_earth_volumetric_mean_radius_ratio.md
Created: sun_mean_density.md
Created: earth_mean_density.md
Created: sun_to_earth_mean_density_ratio.md
Created: sun_surface_gravity.md
Created: earth_surface_gravity.md
Created: sun_to_earth_surface_gravity_ratio.md
Created: sun_escape_velocity.md
Created: earth_escape_velocity.md
Created: sun_to_earth_escape_velocity_ratio.md
Created: sun_ellipticity.md
Created: earth_ellipticity.md
Created: sun_to_earth_ellipticity_ratio.md
Created: sun_moment_of_inertia.md
Created: earth_moment_of_inertia.md
Created: sun_to_earth_moment_of_inertia_ratio.md
Created: sun_visual_magnitude.md
Created: earth_visual_magnitude.md
Created: